In [5]:
# Import libraries
import requests
from bs4 import BeautifulSoup
import dateutil.parser as dparser

# URL from which pdfs to be downloaded
url = "https://official.nba.com/nba-injury-report-2024-25-season/"

# Requests URL and get response object
response = requests.get(url)

# Parse text obtained
soup = BeautifulSoup(response.text, 'html.parser')

# Find all hyperlinks present on webpage
links = soup.find_all('a')

readings_time = {}; readings_pdf = {}
for link in links:
    if link.decode_contents().endswith('ET report'):
        readings_time[link.contents[0]] = dparser.parse(link.contents[0], fuzzy=True, ignoretz=True)
        readings_pdf[link.contents[0]] = requests.get(link.get('href'))

pdf = open('injury.pdf', 'wb')
pdf.write(readings_pdf[max(readings_time, key = readings_time.get)].content)
pdf.close()

In [296]:
import pandas as pd
import numpy as np
import tabula
from dataHub import dataHub
from sqlalchemy.dialects.postgresql.base import PGDialect
PGDialect._get_server_version_info = lambda *args: (9, 2)
dh = dataHub()

db_con = dh.db_connect('cockroach')

top = 75
left = 19 
width = 804
height = 438

dfs = tabula.read_pdf('injury.pdf', area=[top, left, top+height, left+width], pages='all')

status_true = pd.DataFrame({'Current Status': ['Available', 'Probable', 'Questionable', 'Out']})
teams_true = pd.read_sql("SELECT CONCAT(team_long, ' ', team_name) AS Team, team_slug FROM nba.teams", db_con).rename({'team': 'Team'}, axis='columns')

In [297]:
def str_search(df, str_pat, col_ix, col_name):

    if (False in [str_pat in el for el in df.iloc[:, col_ix] if el is not np.nan]):
        df.insert(loc=col_ix, column=col_name, value=np.nan)
    else:
        df.columns.values[col_ix] = col_name

    return df


def occ_count_search(df, df_true, col_ix, col_name):

    df_search = pd.DataFrame({col_name: [el for el in df.iloc[:, col_ix] if el is not np.nan]}).value_counts().reset_index()
     
    if (pd.merge(df_true, df_search, on=col_name, how='left')['count'].sum() == 0):
       df.insert(loc=col_ix, column=col_name, value=np.nan)
    else:
        df.columns.values[col_ix] = col_name

    return df 

In [298]:
for ix, df in enumerate(dfs):
    
    if ix > 0:

        # Move column names to row, excluding first df
        colnames_temp = ['Unnamed: ' + str(el) for el in list(range(0, len(df.columns)))]
        df = (pd.concat([
            pd.DataFrame({key : np.nan if 'Unnamed' in val else val for key, val in dict(zip(colnames_temp, df.columns)).items()}, index=[0]),
            df.set_axis(colnames_temp, axis=1)
        ]))

        # Column checks
        df = str_search(df, '/', 0, 'Game Date') # first col date search
        df = str_search(df, ':', 1, 'Game Time') # second col time search
        df = str_search(df, '@', 2, 'Matchup') # third col matchup search
        df = occ_count_search(df, teams_true, 3, 'Team') # fourth col test for team
        df = str_search(df, ',', 4, 'Player Name') # fifth col test for player name
        df = occ_count_search(df, status_true, 5, 'Current Status') # sixth col test for status
        df.columns.values[6] = 'Reason' # straigt rename of seventh column

    # assign back to index in list
    dfs[ix] = df

# Merge all and clean
df = pd.concat(dfs).reset_index(drop=True)
df['Game Date'] = df['Game Date'].ffill()
df['Game Time'] = df['Game Time'].ffill()
df['Matchup'] = df['Matchup'].ffill()
df['Team'] = df['Team'].ffill()
df['Reason_lead'] = df['Reason'].shift(-1)

# Fix poorly formatted injury rows
inj_ix = df[(df['Reason_lead'].isna()) & (df['Reason'].str.startswith('Injury/Illness'))].index
for ix in inj_ix:

    # Rows of interest
    df_temp = df.iloc[range(ix, ix+3), :]

    # Overwrite rows
    df.loc[range(ix, ix+3), 'Player Name'] = [' '.join(df_temp['Player Name'].dropna())]*3
    df.loc[range(ix, ix+3), 'Current Status'] = [' '.join(df_temp['Current Status'].dropna())]*3
    df.loc[range(ix, ix+3), 'Reason'] = [' '.join(df_temp['Reason'].dropna())]*3

# Drop duplicates and non-submissions
df = df.drop('Reason_lead', axis='columns').drop_duplicates().query('Reason != "NOT YET SUBMITTED"')

# Convert columns
df['Game Time'] = pd.to_datetime(df['Game Date'] + ' ' + df['Game Time'], format='%m/%d/%Y %H:%M (ET)').dt.tz_localize('US/Eastern')
df['Game Date'] = pd.to_datetime(df['Game Date'], format='%m/%d/%Y')
df['Player Name'] = [el[1] + ' ' + el[0] for el in df['Player Name'].str.split(', ')]

# Join external dataset
df.merge(teams_true, how = 'left', on='Team') # team slug
# game id
# player id

# Rename cols
# df = df.rename({'Game Date': 'game_date', 'Game Time': 'game_time', 'Matchup': 'matchup', 'Team': 'team', 'Player Name': 'player_name', 'Current Status': 'status', 'Reason': 'reason'}, axis='columns')

,Game Date,Game Time,Matchup,Team,Player Name,Current Status,Reason,team_slug
0,2025-03-15,2025-03-15 06:00:00-04:00,BOS@BKN,Brooklyn Nets,Nic Claxton,Out,Rest,BKN
1,2025-03-15,2025-03-15 06:00:00-04:00,BOS@BKN,Brooklyn Nets,Noah Clowney,Out,Injury/Illness - Right Ankle; Sprain,BKN
2,2025-03-15,2025-03-15 06:00:00-04:00,BOS@BKN,Brooklyn Nets,Tyson Etienne,Out,G League - Two-Way,BKN
3,2025-03-15,2025-03-15 06:00:00-04:00,BOS@BKN,Brooklyn Nets,Tosan Evbuomwan,Out,G League - Two-Way,BKN
4,2025-03-15,2025-03-15 06:00:00-04:00,BOS@BKN,Brooklyn Nets,De'Anthony Melton,Out,Injury/Illness - Left Knee; ACL Tear,BKN
5,2025-03-15,2025-03-15 06:00:00-04:00,BOS@BKN,Brooklyn Nets,Cam Thomas,Out,Injury/Illness - Left Hamstring; Injury Manage...,BKN
6,2025-03-15,2025-03-15 06:00:00-04:00,BOS@BKN,Brooklyn Nets,Trendon Watford,Out,Injury/Illness - Left Hamstring; Injury Manage...,BKN
7,2025-03-15,2025-03-15 06:00:00-04:00,BOS@BKN,Brooklyn Nets,Dariq Whitehead,Out,G League - On Assignment,BKN
8,2025-03-15,2025-03-15 07:00:00-04:00,OKC@DET,Oklahoma City Thunder,Branden Carlson,Out,G League - Two-Way,OKC
9,2025-03-15,2025-03-15 07:00:00-04:00,OKC@DET,Oklahoma City Thunder,Alex Ducas,Out,G League - Two-Way,OKC


In [299]:
df.merge(teams_true, how = 'left', on='Team')

,Game Date,Game Time,Matchup,Team,Player Name,Current Status,Reason,team_slug
0,2025-03-15,2025-03-15 06:00:00-04:00,BOS@BKN,Brooklyn Nets,Nic Claxton,Out,Rest,BKN
1,2025-03-15,2025-03-15 06:00:00-04:00,BOS@BKN,Brooklyn Nets,Noah Clowney,Out,Injury/Illness - Right Ankle; Sprain,BKN
2,2025-03-15,2025-03-15 06:00:00-04:00,BOS@BKN,Brooklyn Nets,Tyson Etienne,Out,G League - Two-Way,BKN
3,2025-03-15,2025-03-15 06:00:00-04:00,BOS@BKN,Brooklyn Nets,Tosan Evbuomwan,Out,G League - Two-Way,BKN
4,2025-03-15,2025-03-15 06:00:00-04:00,BOS@BKN,Brooklyn Nets,De'Anthony Melton,Out,Injury/Illness - Left Knee; ACL Tear,BKN
5,2025-03-15,2025-03-15 06:00:00-04:00,BOS@BKN,Brooklyn Nets,Cam Thomas,Out,Injury/Illness - Left Hamstring; Injury Manage...,BKN
6,2025-03-15,2025-03-15 06:00:00-04:00,BOS@BKN,Brooklyn Nets,Trendon Watford,Out,Injury/Illness - Left Hamstring; Injury Manage...,BKN
7,2025-03-15,2025-03-15 06:00:00-04:00,BOS@BKN,Brooklyn Nets,Dariq Whitehead,Out,G League - On Assignment,BKN
8,2025-03-15,2025-03-15 07:00:00-04:00,OKC@DET,Oklahoma City Thunder,Branden Carlson,Out,G League - Two-Way,OKC
9,2025-03-15,2025-03-15 07:00:00-04:00,OKC@DET,Oklahoma City Thunder,Alex Ducas,Out,G League - Two-Way,OKC
